# Inspect STRC Data

Load STRC data to investigate the reverse split and momentum calculation.

In [1]:
import pandas as pd
import plotly.graph_objects as go

In [2]:
# Load the prepared momentum dataset
print("Loading momentum dataset...")
df_all = pd.read_parquet('data/momentum_prepared/momentum_data.parquet')
print(f"Loaded {len(df_all):,} rows")

Loading momentum dataset...
Loaded 13,495,461 rows


In [3]:
# Filter for STRC only
df_strc = df_all[df_all['ticker'] == 'STRC'].copy()
df_strc = df_strc.sort_values('date')

print(f"STRC data points: {len(df_strc)}")
print(f"Date range: {df_strc['date'].min().date()} to {df_strc['date'].max().date()}")

STRC data points: 725
Date range: 2021-09-27 to 2025-12-04


In [4]:
# Show all STRC data with key columns
df_strc[[
    'date', 'close', 'adj_close', 'volume', 'adj_volume',
    'price_lag126', 'momentum_6m', 'avg_price_6m', 
    'avg_dollar_volume_1m', 'is_eligible', 'momentum_rank'
]]

,date,close,adj_close,volume,adj_volume,price_lag126,momentum_6m,avg_price_6m,avg_dollar_volume_1m,is_eligible,momentum_rank
11461766,2021-09-27,9.10,54.60,567418,94569.666667,NaN,NaN,NaN,NaN,False,NaN
11461767,2021-09-28,8.95,53.70,481673,80278.833333,NaN,NaN,NaN,NaN,False,NaN
11461768,2021-09-29,7.72,46.32,537045,89507.500000,NaN,NaN,NaN,NaN,False,NaN
11461769,2021-09-30,7.73,46.38,234760,39126.666667,NaN,NaN,NaN,NaN,False,NaN
11461770,2021-10-01,7.34,44.04,225856,37642.666667,NaN,NaN,NaN,NaN,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...
11462486,2025-11-28,96.66,96.66,523464,523464.000000,0.4750,202.494737,66.954809,1.419625e+08,True,1.0
11462487,2025-12-01,96.41,96.41,988476,988476.000000,0.4951,193.728338,67.716038,1.418340e+08,True,1.0
11462488,2025-12-02,97.80,97.80,780921,780921.000000,0.5300,183.528302,68.488022,1.386112e+08,True,1.0
11462489,2025-12-03,98.20,98.20,597053,597053.000000,0.4812,203.073150,69.263568,1.355662e+08,True,1.0


In [5]:
# Load split data
splits_df = pd.read_csv('data/splits/splits_data.csv')
strc_splits = splits_df[splits_df['ticker'] == 'STRC']

print("STRC Splits:")
strc_splits

STRC Splits:


,ticker,execution_date,split_from,split_to,split_ratio
2170,STRC,2023-07-06,6.0,1.0,0.166667


In [6]:
# Show data around the split date (2023-07-06)
split_date = pd.to_datetime('2023-07-06')
around_split = df_strc[
    (df_strc['date'] >= split_date - pd.Timedelta(days=10)) &
    (df_strc['date'] <= split_date + pd.Timedelta(days=10))
].copy()

print(f"\nData around reverse split date (2023-07-06):")
around_split[[
    'date', 'close', 'adj_close', 'volume', 'adj_volume'
]]


Data around reverse split date (2023-07-06):


,date,close,adj_close,volume,adj_volume
11462204,2023-06-26,0.3700,2.2200,875057,145842.833333
11462205,2023-06-27,0.3407,2.0442,617691,102948.500000
11462206,2023-06-28,0.3400,2.0400,566540,94423.333333
11462207,2023-06-29,0.3315,1.9890,454374,75729.000000
11462208,2023-06-30,0.3215,1.9290,694871,115811.833333
11462209,2023-07-03,0.2999,1.7994,881669,146944.833333
11462210,2023-07-05,0.3170,1.9020,821330,136888.333333
11462211,2023-07-06,1.6000,1.6000,254873,254873.000000
11462212,2023-07-07,1.5800,1.5800,78343,78343.000000
11462213,2023-07-10,1.5100,1.5100,236714,236714.000000


In [7]:
# Chart the full history
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_strc['date'],
    y=df_strc['adj_close'],
    mode='lines',
    name='Adjusted Close',
    line=dict(color='blue', width=2)
))

# Add vertical line at split date
fig.add_vline(
    x=split_date,
    line_dash="dash",
    line_color="red",
    annotation_text="6-for-1 Reverse Split",
    annotation_position="top"
)

fig.update_layout(
    title='STRC - Split-Adjusted Price History',
    xaxis_title='Date',
    yaxis_title='Adjusted Close Price ($)',
    height=600,
    hovermode='x unified'
)

fig.show()

TypeError: Addition/subtraction of integers and integer-arrays with Timestamp is no longer supported.  Instead of adding/subtracting `n`, use `n * obj.freq`

In [ ]:
# Show recent data with momentum
recent = df_strc.tail(200)

print("\nRecent STRC data (last 200 days):")
recent[[
    'date', 'adj_close', 'price_lag126', 'momentum_6m', 
    'is_eligible', 'momentum_rank'
]]

In [ ]:
# Analyze the momentum calculation for latest date
latest = df_strc.iloc[-1]

print("\n" + "="*70)
print("STRC Latest Data Analysis")
print("="*70)
print(f"Date: {latest['date'].date()}")
print(f"Current adj_close: ${latest['adj_close']:.2f}")
print(f"Price 126 days ago: ${latest['price_lag126']:.2f}")
print(f"Momentum calculation: ({latest['adj_close']:.2f} - {latest['price_lag126']:.2f}) / {latest['price_lag126']:.2f}")
print(f"Momentum: {latest['momentum_6m']:.2%}")
print(f"Is eligible: {latest['is_eligible']}")
print(f"Momentum rank: {latest['momentum_rank']}")
print(f"\nAvg 6M price: ${latest['avg_price_6m']:.2f}")
print(f"Avg monthly $ volume: ${latest['avg_dollar_volume_1m']:,.0f}")